In [3]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
import io
import sys
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import seaborn as sns
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score, f1_score
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from IPython.display import display

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score

from typing import Tuple
from sklearn.base import RegressorMixin
from typing import Tuple, List
from sklearn.pipeline import Pipeline
from sklearn.base import RegressorMixin
from typing import Optional
from sklearn.base import ClassifierMixin

from xgboost import XGBClassifier
from sklearn.ensemble import StackingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

from typing import Tuple, List, Dict
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_curve, auc
from sklearn.metrics import precision_recall_curve, average_precision_score

from sklearn.pipeline import make_pipeline
import optuna.visualization as vis
from statsmodels.tsa.arima.model import ARIMA


from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX

import textwrap, pathlib, json, sys, inspect, os, types, importlib, pathlib, matplotlib, pandas as pd, numpy as np




In [2]:
module_code = """
\"\"\"ts_automator.py
Simple utility to automate univariate time‑series analysis and basic forecasting.
Author: ChatGPT (@OpenAI)
Created: 2025‑04‑25
\"\"\"

from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

import pathlib
from typing import Union, Tuple, Optional

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Optional (soft) dependencies
try:
    import pmdarima as pm
except ImportError:  # pragma: no cover
    pm = None

try:
    from statsmodels.tsa.seasonal import seasonal_decompose
except ImportError:  # pragma: no cover
    seasonal_decompose = None

###############################################################################
# Helpers
###############################################################################
def _load_series(
    data: Union[str, pd.DataFrame],
    date_col: str,
    target_col: str,
    freq: Optional[str] = None,
) -> pd.Series:
    \"\"\"Load a CSV or DataFrame and return a pd.Series indexed by DatetimeIndex.\"\"\"
    if isinstance(data, str):
        df = pd.read_csv(data)
    elif isinstance(data, pd.DataFrame):
        df = data.copy()
    else:
        raise TypeError(\"`data` must be path to csv or a pandas.DataFrame\")

    if date_col not in df.columns or target_col not in df.columns:
        raise ValueError(\"`date_col` and `target_col` must exist in dataframe\")

    df[date_col] = pd.to_datetime(df[date_col], errors=\"coerce\")
    df = df.set_index(date_col).sort_index()

    if freq is not None:
        df = df.asfreq(freq)

    return df[target_col].dropna()


def _simple_plot(serie: pd.Series, title: str = \"Time‑series plot\", **_) -> None:
    serie.plot(figsize=(10, 4))
    plt.title(title)
    plt.tight_layout()
    plt.show()


def _decompose(
    serie: pd.Series,
    model: str = \"additive\",
    seasonal_period: Optional[int] = None,
    **_,
) -> None:
    if seasonal_decompose is None:
        raise ImportError(\"statsmodels is required for decomposition\")
    result = seasonal_decompose(serie, model=model, period=seasonal_period)
    result.plot()
    plt.tight_layout()
    plt.show()


def _auto_arima_forecast(
    serie: pd.Series,
    n_periods: int = 24,
    seasonal: bool = False,
    m: int = 1,
    **_,
) -> Tuple[pd.Series, pd.DataFrame]:
    if pm is None:
        raise ImportError(\"pmdarima is required for auto_arima functionality\")
    model = pm.auto_arima(
        serie,
        seasonal=seasonal,
        m=m,
        error_action=\"ignore\",
        suppress_warnings=True,
        stepwise=True,
    )
    fc, conf = model.predict(n_periods=n_periods, return_conf_int=True)
    idx_fc = pd.date_range(
        serie.index[-1],
        periods=n_periods + 1,
        freq=serie.index.freq or pd.infer_freq(serie.index),
    )[1:]
    fc_series = pd.Series(fc, index=idx_fc)
    conf_df = pd.DataFrame(conf, index=idx_fc, columns=[\"lower\", \"upper\"])

    # Plot
    plt.figure(figsize=(10, 4))
    serie.plot(label=\"historical\")
    fc_series.plot(label=\"forecast\")
    plt.fill_between(conf_df.index, conf_df[\"lower\"], conf_df[\"upper\"], alpha=0.3)
    plt.title(\"auto_arima forecast\")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return fc_series, conf_df


###############################################################################
# Public API
###############################################################################
def run_timeseries(
    data: Union[str, pd.DataFrame],
    date_col: str,
    target_col: str,
    method: str = \"plot\",
    freq: Optional[str] = None,
    **kwargs,
):
    \"\"\"Run a selected workflow.

    Parameters
    ----------
    data
        Path to CSV file or DataFrame containing the data.
    date_col
        Column with timestamp information.
    target_col
        Column with the numeric series to analyze.
    method
        One of {\"plot\", \"decompose\", \"auto_arima\"}.
    freq
        Optional pandas offset string to enforce a regular frequency (e.g., \"MS\").
    kwargs
        Additional parameters forwarded to the underlying workflow functions.

    Returns
    -------
    Depends on the method. For forecasting methods, returns forecast outputs.
    \"\"\"
    serie = _load_series(data, date_col=date_col, target_col=target_col, freq=freq)

    method = method.lower()
    if method == \"plot\":
        _simple_plot(serie, **kwargs)
    elif method == \"decompose\":
        _decompose(serie, **kwargs)
    elif method == \"auto_arima\":
        return _auto_arima_forecast(serie, **kwargs)
    else:
        raise ValueError(f\"Unknown method: {method}\")
"""

# Write the module to disk
module_path = pathlib.Path('/mnt/data/ts_automator.py')
module_path.write_text(textwrap.dedent(module_code))

print(f"Module written to {module_path}")




usage: ipykernel_launcher.py [-h] [--method METHOD] [--freq FREQ]
                             [--n_periods N_PERIODS]
                             data date_col target_col
ipykernel_launcher.py: error: the following arguments are required: date_col, target_col


SystemExit: 2

In [3]:
import sys, pathlib
sys.path.append('/mnt/data')          # Añade la carpeta donde se guardó
import ts_automator as tsa            # ¡listo!

# Ejemplo rápido:
tsa.run_timeseries(
    'Registros_Condiciones.csv',
    date_col='fecha',
    target_col='temperatura',
    method='plot',
    freq='H'      # ajusta frecuencia si lo deseas
)


ModuleNotFoundError: No module named 'ts_automator'